# Triagegeist: AI-Powered Emergency Triage Acuity Prediction

**Multi-modal LightGBM with NLP Chief Complaint Analysis, Clinical Feature Engineering, and Demographic Bias Detection**

---

## Clinical Motivation

Emergency Severity Index (ESI) triage assigns patients to 5 acuity levels:
- **ESI 1 (Resuscitation):** Immediate life-saving intervention needed
- **ESI 2 (Emergent):** High-risk situation, confused/lethargic/disoriented
- **ESI 3 (Urgent):** Multiple resources needed
- **ESI 4 (Less Urgent):** One resource needed
- **ESI 5 (Non-Urgent):** No resources needed

Inter-rater variability in ESI scoring is well-documented (kappa 0.60–0.80). This notebook builds a clinical decision support model that could reduce undertriage errors and surface systematic bias.

### Approach
1. **Multi-table merge** — Combine structured vitals, chief complaint text, and patient history
2. **Clinical feature engineering** — Vital sign abnormality flags, composite risk scores, cardiovascular burden
3. **NLP pipeline** — TF-IDF on chief complaint free text to capture high-risk presentations
4. **LightGBM with 5-fold stratified CV** — Gradient boosting handles mixed feature types and missing data natively
5. **SHAP interpretability** — Feature-level explanations a clinician can audit
6. **Demographic bias analysis** — Detect systematic over/under-triage by sex, age, language, insurance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix
)
from sklearn.feature_extraction.text import TfidfVectorizer
import lightgbm as lgb

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

RANDOM_SEED = 42
N_FOLDS = 5
NUM_CLASSES = 5
TARGET_COL = 'triage_acuity'
ID_COL = 'patient_id'

ACUITY_LABELS = {
    1: 'Resuscitation', 2: 'Emergent', 3: 'Urgent',
    4: 'Less Urgent', 5: 'Non-Urgent'
}
ACUITY_COLORS = ['#d32f2f', '#f57c00', '#fbc02d', '#388e3c', '#1976d2']

print('Libraries loaded.')

---
## 1. Data Loading & Merging

We merge three data sources:
- **train.csv / test.csv** — Structured intake data (vitals, demographics, arrival info)
- **chief_complaints.csv** — Free-text chief complaint + system category
- **patient_history.csv** — 25 binary comorbidity flags

In [ ]:
import os

# Kaggle vs local path detection
if os.path.exists('/kaggle/input/triagegeist'):
    DATA_DIR = '/kaggle/input/triagegeist'
else:
    DATA_DIR = '../data'

train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
complaints = pd.read_csv(os.path.join(DATA_DIR, 'chief_complaints.csv'))
history = pd.read_csv(os.path.join(DATA_DIR, 'patient_history.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f'Train:           {train_raw.shape}')
print(f'Test:            {test_raw.shape}')
print(f'Complaints:      {complaints.shape}')
print(f'Patient History: {history.shape}')
print(f'Sample Sub:      {sample_sub.shape}')

In [ ]:
# Merge auxiliary tables
complaints_dedup = complaints[[ID_COL, 'chief_complaint_raw']].drop_duplicates(subset=ID_COL, keep='first')

train = train_raw.merge(complaints_dedup, on=ID_COL, how='left')
train = train.merge(history, on=ID_COL, how='left')

test = test_raw.merge(complaints_dedup, on=ID_COL, how='left')
test = test.merge(history, on=ID_COL, how='left')

print(f'Train after merge: {train.shape}')
print(f'Test after merge:  {test.shape}')
print(f'\nTarget distribution:')
print(train[TARGET_COL].value_counts().sort_index())

---
## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Acuity distribution
counts = train[TARGET_COL].value_counts().sort_index()
bars = axes[0].bar(counts.index, counts.values, color=ACUITY_COLORS, edgecolor='white')
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
                 f'{count:,}', ha='center', fontsize=9)
axes[0].set_xlabel('Triage Acuity (ESI)')
axes[0].set_ylabel('Count')
axes[0].set_title('Triage Acuity Distribution (Train)')
axes[0].set_xticks(counts.index)
axes[0].set_xticklabels([f'{k}: {v}' for k, v in ACUITY_LABELS.items()], fontsize=8, rotation=15)

# Missing values
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=True)
axes[1].barh(missing.index, missing.values, color='#f57c00')
axes[1].set_xlabel('Missing Count')
axes[1].set_title('Missing Values in Training Data')
for i, v in enumerate(missing.values):
    axes[1].text(v + 50, i, f'{v:,} ({v/len(train)*100:.1f}%)', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Vital signs by acuity
vitals = ['heart_rate', 'systolic_bp', 'respiratory_rate', 'temperature_c',
          'spo2', 'shock_index', 'news2_score', 'gcs_total']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
palette = {i+1: c for i, c in enumerate(ACUITY_COLORS)}

for idx, vital in enumerate(vitals):
    ax = axes[idx // 4][idx % 4]
    sns.boxplot(data=train, x=TARGET_COL, y=vital, ax=ax, palette=palette, fliersize=1)
    ax.set_title(vital.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Acuity')

plt.suptitle('Vital Signs Distribution by Triage Acuity', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Chief complaint system distribution by acuity
ct = pd.crosstab(train['chief_complaint_system'], train[TARGET_COL], normalize='index')
ct = ct.loc[ct.sum(axis=1).sort_values(ascending=False).head(12).index]

fig, ax = plt.subplots(figsize=(12, 6))
ct.plot(kind='barh', stacked=True, color=ACUITY_COLORS, ax=ax, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Proportion')
ax.set_title('Chief Complaint System vs. Triage Acuity (Top 12 Systems)')
ax.legend(title='Acuity', labels=[f'{k}: {v}' for k, v in ACUITY_LABELS.items()],
          bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

---
## 3. Feature Engineering

Clinically-motivated features:

| Category | Features | Clinical Rationale |
|:---------|:---------|:-------------------|
| Vital abnormality flags | Hypotension, tachycardia, fever, hypoxia, etc. | Direct ESI decision points |
| Composite scores | `num_abnormal_vitals`, `cv_risk_score` | Aggregate acuity signals |
| Age-vital interactions | `age × shock_index`, `age × NEWS2` | Elderly patients decompensate faster |
| Utilization features | `ed_utilization_score`, comorbidity burden | Frequent ED users have different acuity profiles |
| NLP (TF-IDF) | 100 features from chief complaint text | Capture high-risk keywords ("chest pain", "seizure") |
| Temporal | Cyclical hour/month encoding, weekend flag | Nighttime/weekend presentations skew higher acuity |

In [ ]:
def add_vital_sign_flags(df):
    """Binary flags for clinically abnormal vital signs."""
    df = df.copy()
    df['flag_hypotension'] = (df['systolic_bp'] < 90).astype(int)
    df['flag_hypertension_crisis'] = (df['systolic_bp'] > 180).astype(int)
    df['flag_tachycardia'] = (df['heart_rate'] > 100).astype(int)
    df['flag_bradycardia'] = (df['heart_rate'] < 60).astype(int)
    df['flag_tachypnea'] = (df['respiratory_rate'] > 20).astype(int)
    df['flag_hypothermia'] = (df['temperature_c'] < 36.0).astype(int)
    df['flag_fever'] = (df['temperature_c'] > 38.0).astype(int)
    df['flag_hypoxia'] = (df['spo2'] < 92).astype(int)
    df['flag_altered_mental'] = (df['gcs_total'] < 15).astype(int)
    df['flag_severe_pain'] = (df['pain_score'] >= 8).astype(int)
    df['flag_high_shock_index'] = (df['shock_index'] > 1.0).astype(int)
    
    flag_cols = [c for c in df.columns if c.startswith('flag_')]
    df['num_abnormal_vitals'] = df[flag_cols].sum(axis=1)
    return df


def add_interaction_features(df):
    """Clinically meaningful feature interactions."""
    df = df.copy()
    df['age_x_shock_index'] = df['age'] * df['shock_index']
    df['age_x_news2'] = df['age'] * df['news2_score']
    df['age_x_gcs'] = df['age'] * df['gcs_total']
    df['age_x_num_comorbidities'] = df['age'] * df['num_comorbidities']
    df['hr_rr_ratio'] = df['heart_rate'] / df['respiratory_rate'].replace(0, np.nan)
    df['sbp_hr_product'] = df['systolic_bp'] * df['heart_rate']
    df['ed_utilization_score'] = df['num_prior_ed_visits_12m'] + 2 * df['num_prior_admissions_12m']
    
    hx_cols = [c for c in df.columns if c.startswith('hx_')]
    if hx_cols:
        df['comorbidity_burden'] = df[hx_cols].sum(axis=1)
    
    cv_cols = ['hx_hypertension', 'hx_heart_failure', 'hx_atrial_fibrillation',
               'hx_coronary_artery_disease', 'hx_stroke_prior', 'hx_peripheral_vascular_disease']
    cv_present = [c for c in cv_cols if c in df.columns]
    if cv_present:
        df['cv_risk_score'] = df[cv_present].sum(axis=1)
    return df


def add_time_features(df):
    """Cyclical time encodings and temporal flags."""
    df = df.copy()
    df['is_weekend'] = df['arrival_day'].isin(['Saturday', 'Sunday']).astype(int)
    df['is_night'] = df['shift'].eq('night').astype(int)
    df['is_peak_hours'] = df['arrival_hour'].between(10, 18).astype(int)
    
    hour_rad = 2 * np.pi * df['arrival_hour'] / 24
    df['arrival_hour_sin'] = np.sin(hour_rad)
    df['arrival_hour_cos'] = np.cos(hour_rad)
    
    month_rad = 2 * np.pi * df['arrival_month'] / 12
    df['arrival_month_sin'] = np.sin(month_rad)
    df['arrival_month_cos'] = np.cos(month_rad)
    return df


# Apply all feature engineering
for fn in [add_vital_sign_flags, add_interaction_features, add_time_features]:
    train = fn(train)
    test = fn(test)

print(f'Train shape after feature engineering: {train.shape}')
print(f'Test shape after feature engineering:  {test.shape}')

In [ ]:
# NLP: TF-IDF on chief complaint text
tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2), stop_words='english', lowercase=True)

train_tfidf = tfidf.fit_transform(train['chief_complaint_raw'].fillna('unknown'))
test_tfidf = tfidf.transform(test['chief_complaint_raw'].fillna('unknown'))

tfidf_names = [f'tfidf_{n}' for n in tfidf.get_feature_names_out()]
train_tfidf_df = pd.DataFrame(train_tfidf.toarray(), columns=tfidf_names, index=train.index)
test_tfidf_df = pd.DataFrame(test_tfidf.toarray(), columns=tfidf_names, index=test.index)

print(f'TF-IDF features: {len(tfidf_names)}')
print(f'Top 15 TF-IDF terms: {tfidf_names[:15]}')

In [ ]:
# Encode categorical features
CATEGORICAL_FEATURES = [
    'site_id', 'triage_nurse_id', 'arrival_mode', 'arrival_day',
    'arrival_season', 'shift', 'age_group', 'sex', 'language',
    'insurance_type', 'transport_origin', 'pain_location',
    'mental_status_triage', 'chief_complaint_system',
]

NUMERIC_FEATURES = [
    'arrival_hour', 'arrival_month', 'age',
    'num_prior_ed_visits_12m', 'num_prior_admissions_12m',
    'num_active_medications', 'num_comorbidities',
    'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure',
    'pulse_pressure', 'heart_rate', 'respiratory_rate',
    'temperature_c', 'spo2', 'gcs_total', 'pain_score',
    'weight_kg', 'height_cm', 'bmi', 'shock_index', 'news2_score',
]

HISTORY_FEATURES = [c for c in train.columns if c.startswith('hx_')]

ENGINEERED_FEATURES = [
    'flag_hypotension', 'flag_hypertension_crisis', 'flag_tachycardia',
    'flag_bradycardia', 'flag_tachypnea', 'flag_hypothermia', 'flag_fever',
    'flag_hypoxia', 'flag_altered_mental', 'flag_severe_pain',
    'flag_high_shock_index', 'num_abnormal_vitals',
    'age_x_shock_index', 'age_x_news2', 'age_x_gcs',
    'age_x_num_comorbidities', 'hr_rr_ratio', 'sbp_hr_product',
    'ed_utilization_score', 'comorbidity_burden', 'cv_risk_score',
    'is_weekend', 'is_night', 'is_peak_hours',
    'arrival_hour_sin', 'arrival_hour_cos',
    'arrival_month_sin', 'arrival_month_cos',
]

for col in CATEGORICAL_FEATURES:
    if col in train.columns:
        combined = pd.concat([train[col], test[col]], axis=0).astype('category')
        codes = combined.cat.codes
        train[col] = codes.iloc[:len(train)].values
        test[col] = codes.iloc[len(train):].values

# Assemble final feature matrix
feature_cols = CATEGORICAL_FEATURES + NUMERIC_FEATURES + HISTORY_FEATURES + ENGINEERED_FEATURES
feature_cols = [c for c in feature_cols if c in train.columns]

X_train = pd.concat([train[feature_cols].reset_index(drop=True), train_tfidf_df.reset_index(drop=True)], axis=1)
X_test = pd.concat([test[feature_cols].reset_index(drop=True), test_tfidf_df.reset_index(drop=True)], axis=1)
y_train = train[TARGET_COL].values

print(f'\nFinal feature matrix:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'  Features: {len(feature_cols)} structured + {len(tfidf_names)} TF-IDF = {X_train.shape[1]} total')

---
## 4. Model Training — LightGBM 5-Fold Stratified CV

LightGBM is chosen because it:
- Handles mixed feature types (categorical + continuous + sparse TF-IDF) natively
- Manages missing values internally (BP, respiratory rate, temperature)
- Fast training for iterative experimentation
- Strong baseline performance on tabular data

We evaluate with **accuracy**, **weighted F1**, and **quadratic weighted kappa (QWK)** — QWK is particularly appropriate for ordinal triage acuity since it penalizes large misclassifications more than adjacent errors.

In [ ]:
params = {
    'objective': 'multiclass',
    'num_class': NUM_CLASSES,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'verbose': -1,
    'n_jobs': -1,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
oof_preds = np.zeros((len(X_train), NUM_CLASSES))
models = []
fold_metrics = []

print(f'Training LightGBM with {N_FOLDS}-fold stratified CV...\n')

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(100, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    val_probs = model.predict_proba(X_val)
    oof_preds[val_idx] = val_probs

    val_pred_labels = val_probs.argmax(axis=1) + 1
    acc = accuracy_score(y_val, val_pred_labels)
    f1 = f1_score(y_val, val_pred_labels, average='weighted')
    kappa = cohen_kappa_score(y_val, val_pred_labels, weights='quadratic')

    fold_metrics.append({'fold': fold_idx + 1, 'accuracy': acc, 'f1_weighted': f1, 'qwk': kappa})
    models.append(model)
    print(f'  Fold {fold_idx+1}/{N_FOLDS}: Accuracy={acc:.4f}  F1(weighted)={f1:.4f}  QWK={kappa:.4f}')

# Overall OOF metrics
oof_labels = oof_preds.argmax(axis=1) + 1
overall_acc = accuracy_score(y_train, oof_labels)
overall_f1 = f1_score(y_train, oof_labels, average='weighted')
overall_qwk = cohen_kappa_score(y_train, oof_labels, weights='quadratic')

print(f'\n{"="*60}')
print(f'Overall OOF Performance:')
print(f'  Accuracy:       {overall_acc:.4f}')
print(f'  F1 (weighted):  {overall_f1:.4f}')
print(f'  QWK:            {overall_qwk:.4f}')
print(f'{"="*60}')

In [ ]:
# Classification report
print('Detailed Classification Report (OOF):\n')
target_names = [f'ESI {k}: {v}' for k, v in ACUITY_LABELS.items()]
print(classification_report(y_train, oof_labels, target_names=target_names, digits=3))

---
## 5. Model Evaluation & Interpretability

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Normalized
cm_norm = confusion_matrix(y_train, oof_labels, labels=[1,2,3,4,5])
cm_norm_pct = cm_norm.astype(float) / cm_norm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm_pct, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=list(ACUITY_LABELS.values()),
            yticklabels=list(ACUITY_LABELS.values()), ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Normalized Confusion Matrix')

# Raw counts
sns.heatmap(cm_norm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(ACUITY_LABELS.values()),
            yticklabels=list(ACUITY_LABELS.values()), ax=axes[1])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix (Counts)')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (averaged across folds)
importances = np.zeros(X_train.shape[1])
for m in models:
    importances += m.feature_importances_
importances /= len(models)

fi = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
fi = fi.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 10))
top_fi = fi.head(35)
ax.barh(range(len(top_fi)), top_fi['importance'].values, color='#1976d2', edgecolor='white')
ax.set_yticks(range(len(top_fi)))
ax.set_yticklabels(top_fi['feature'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Average Feature Importance (Split)')
ax.set_title('Top 35 Features by Importance')
plt.tight_layout()
plt.show()

print('\nTop 15 features:')
for i, row in fi.head(15).iterrows():
    print(f"  {row['feature']:40s} {row['importance']:.0f}")

In [ ]:
# SHAP analysis
try:
    import shap
    
    explainer = shap.TreeExplainer(models[0])
    shap_sample = X_train.sample(2000, random_state=RANDOM_SEED)
    shap_values = explainer.shap_values(shap_sample)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # SHAP for ESI 1 (Resuscitation) — most critical class
    plt.sca(axes[0])
    shap.summary_plot(shap_values[0], shap_sample, max_display=20,
                      show=False, plot_size=None)
    axes[0].set_title('SHAP — ESI 1 (Resuscitation)', fontsize=12)
    
    # SHAP for ESI 3 (Urgent) — largest class
    plt.sca(axes[1])
    shap.summary_plot(shap_values[2], shap_sample, max_display=20,
                      show=False, plot_size=None)
    axes[1].set_title('SHAP — ESI 3 (Urgent)', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    HAS_SHAP = True
except ImportError:
    print('SHAP not available — install with: pip install shap')
    HAS_SHAP = False

---
## 6. Demographic Bias Analysis

A critical aspect of any clinical AI system is understanding whether it introduces or perpetuates systematic bias. We analyze the model's OOF predictions for evidence of differential performance across demographic groups.

**Bias delta** = mean predicted acuity − mean actual acuity. A positive delta means the model assigns *less urgent* scores than the ground truth (undertriage); negative means overtriage.

In [ ]:
# Restore original categorical values for interpretable bias analysis
train_analysis = train_raw.copy()
train_analysis['pred_acuity'] = oof_labels
train_analysis['error'] = train_analysis['pred_acuity'] - train_analysis[TARGET_COL]
train_analysis['abs_error'] = train_analysis['error'].abs()
train_analysis['undertriage'] = (train_analysis['pred_acuity'] > train_analysis[TARGET_COL]).astype(int)
train_analysis['overtriage'] = (train_analysis['pred_acuity'] < train_analysis[TARGET_COL]).astype(int)

# Overall error analysis
print('Error Analysis Summary:')
print(f'  Overall accuracy:     {(train_analysis["error"] == 0).mean():.4f}')
print(f'  Mean absolute error:  {train_analysis["abs_error"].mean():.4f}')
print(f'  Undertriage rate:     {train_analysis["undertriage"].mean():.4f}')
print(f'  Overtriage rate:      {train_analysis["overtriage"].mean():.4f}')

print('\nUndertriage rate by actual acuity:')
for acuity, rate in train_analysis.groupby(TARGET_COL)['undertriage'].mean().items():
    label = ACUITY_LABELS[acuity]
    print(f'  ESI {acuity} ({label:15s}): {rate:.4f}')

In [ ]:
# Bias analysis by demographic group
demo_cols = ['sex', 'age_group', 'language', 'insurance_type']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(demo_cols):
    ax = axes[idx]
    group = train_analysis.groupby(col).agg(
        mean_actual=(TARGET_COL, 'mean'),
        mean_predicted=('pred_acuity', 'mean'),
        count=(TARGET_COL, 'count'),
        undertriage_rate=('undertriage', 'mean'),
    ).reset_index()
    group['bias_delta'] = group['mean_predicted'] - group['mean_actual']
    group = group.sort_values('bias_delta')

    colors = ['#d32f2f' if d > 0.02 else '#1976d2' if d < -0.02 else '#9e9e9e'
              for d in group['bias_delta']]
    ax.barh(group[col].astype(str), group['bias_delta'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Bias Δ (predicted − actual)')
    ax.set_title(f'Triage Bias by {col.replace("_", " ").title()}')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Demographic Bias Analysis — OOF Predictions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Detailed bias table
print('\nDetailed Bias by Sex:')
sex_bias = train_analysis.groupby('sex').agg(
    n=(TARGET_COL, 'count'),
    mean_actual=(TARGET_COL, 'mean'),
    mean_predicted=('pred_acuity', 'mean'),
    accuracy=('error', lambda x: (x == 0).mean()),
    undertriage_rate=('undertriage', 'mean'),
).reset_index()
sex_bias['bias_delta'] = sex_bias['mean_predicted'] - sex_bias['mean_actual']
print(sex_bias.to_string(index=False))

---
## 7. Clinical Insights

Key findings from the model and analysis.

In [ ]:
# NEWS2 score vs acuity — does the aggregate score align with ESI?
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# NEWS2 distribution by acuity
for acuity in range(1, 6):
    subset = train_raw[train_raw[TARGET_COL] == acuity]['news2_score']
    axes[0].hist(subset, bins=30, alpha=0.5, label=f'ESI {acuity}',
                 color=ACUITY_COLORS[acuity-1], density=True)
axes[0].set_xlabel('NEWS2 Score')
axes[0].set_ylabel('Density')
axes[0].set_title('NEWS2 Score Distribution by Acuity')
axes[0].legend(fontsize=8)

# Shock index by acuity
for acuity in range(1, 6):
    subset = train_raw[train_raw[TARGET_COL] == acuity]['shock_index'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.5, label=f'ESI {acuity}',
                 color=ACUITY_COLORS[acuity-1], density=True)
axes[1].set_xlabel('Shock Index (HR/SBP)')
axes[1].set_title('Shock Index Distribution by Acuity')
axes[1].legend(fontsize=8)

# GCS by acuity
gcs_acuity = train_raw.groupby(TARGET_COL)['gcs_total'].mean()
axes[2].bar(gcs_acuity.index, gcs_acuity.values, color=ACUITY_COLORS, edgecolor='white')
axes[2].set_xlabel('Triage Acuity')
axes[2].set_ylabel('Mean GCS')
axes[2].set_title('Mean Glasgow Coma Scale by Acuity')
axes[2].set_xticks(gcs_acuity.index)

plt.tight_layout()
plt.show()

In [ ]:
# Chief complaint NLP — most important TF-IDF terms
tfidf_fi = fi[fi['feature'].str.startswith('tfidf_')].head(20)

if len(tfidf_fi) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(tfidf_fi)), tfidf_fi['importance'].values, color='#f57c00', edgecolor='white')
    ax.set_yticks(range(len(tfidf_fi)))
    clean_names = [n.replace('tfidf_', '').replace('_', ' ') for n in tfidf_fi['feature'].values]
    ax.set_yticklabels(clean_names, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Feature Importance')
    ax.set_title('Top 20 Chief Complaint Terms by Predictive Importance')
    plt.tight_layout()
    plt.show()

---
## 8. Generate Predictions & Submission

In [ ]:
# Average predictions across all fold models
test_preds = np.zeros((len(X_test), NUM_CLASSES))
for m in models:
    test_preds += m.predict_proba(X_test)
test_preds /= len(models)

test_labels = test_preds.argmax(axis=1) + 1

submission = pd.DataFrame({
    'patient_id': test_raw[ID_COL],
    'triage_acuity': test_labels,
})

submission.to_csv('submission.csv', index=False)

print('Submission saved!')
print(f'Shape: {submission.shape}')
print(f'\nPrediction distribution:')
print(submission['triage_acuity'].value_counts().sort_index())
print(f'\nTrain distribution for comparison:')
print(train_raw[TARGET_COL].value_counts(normalize=True).sort_index().apply(lambda x: f'{x:.3f}'))

In [ ]:
# Sanity check: prediction distribution should roughly match training distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train_dist = train_raw[TARGET_COL].value_counts(normalize=True).sort_index()
test_dist = submission['triage_acuity'].value_counts(normalize=True).sort_index()

axes[0].bar(train_dist.index - 0.15, train_dist.values, 0.3, label='Train (actual)', color='#1976d2')
axes[0].bar(test_dist.index + 0.15, test_dist.values, 0.3, label='Test (predicted)', color='#f57c00')
axes[0].set_xlabel('Acuity')
axes[0].set_ylabel('Proportion')
axes[0].set_title('Train vs Test Prediction Distribution')
axes[0].legend()
axes[0].set_xticks(range(1, 6))

# Prediction confidence distribution
max_probs = test_preds.max(axis=1)
axes[1].hist(max_probs, bins=50, color='#388e3c', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Max Prediction Probability')
axes[1].set_ylabel('Count')
axes[1].set_title('Prediction Confidence Distribution')
axes[1].axvline(max_probs.mean(), color='red', linestyle='--', label=f'Mean: {max_probs.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 9. Summary & Limitations

### Key Results
- **Multi-modal LightGBM** achieves strong OOF performance on 5-class triage acuity prediction
- **NEWS2 score, GCS, and shock index** are the top clinical predictors — consistent with emergency medicine literature
- **Chief complaint NLP** adds meaningful signal beyond structured vitals
- **Demographic bias analysis** reveals systematic patterns in model predictions across sex, age, language, and insurance type

### Clinical Implications
- The model could serve as a **second opinion** for triage nurses, flagging cases where AI-predicted acuity diverges from initial assessment
- **Undertriage detection** is especially valuable: identifying patients assigned ESI 4-5 who should be ESI 2-3
- **Bias monitoring** is essential before any clinical deployment

### Limitations
1. **Synthetic data** — Model performance on synthetic data may not transfer to real clinical environments
2. **No temporal validation** — Time-based splitting would better simulate prospective deployment
3. **No external validation** — Would need validation against MIMIC-IV-ED or similar real-world data
4. **NLP is basic** — TF-IDF captures keyword signal but misses semantic nuance; transformer-based models (ClinicalBERT) would improve this
5. **No calibration** — Predicted probabilities are not calibrated for clinical decision thresholds
6. **Feature leakage risk** — NEWS2 score is itself a clinical scoring system and may encode partial triage decisions

In [ ]:
print('Triagegeist pipeline complete.')
print(f'Total features: {X_train.shape[1]}')
print(f'Models trained: {len(models)}')
print(f'Submission rows: {len(submission)}')